# ALICE Run 3 simulation hands-on — 0. Setup

This notebook checks that your SWAN session can run the ALICE simulation software from CVMFS.

**Session settings** (in the SWAN start page, before starting the session):

| Setting | Value |
|---|---|
| Software stack | latest default LCG release (the notebooks do **not** use it for O2) |
| Platform | AlmaLinux 9 |
| Number of cores | **4** |
| Memory | **16 GB** |
| Environment script | *(leave empty)* |

**How O2 is used here.** Every notebook starts with `from swan_env import *`.
After that line, every cell starting with `%%bash` runs inside the **ALICE O2 environment** loaded from `/cvmfs/alice.cern.ch`, so `o2-sim`, `root` and the O2DPG scripts can be used directly.
Each `%%bash` cell runs in its own shell. Output appears while the cell runs, and the ■ (stop) button stops it.

In [ ]:
from swan_env import *

## 1. Session info

In [ ]:
%%bash
echo "work directory: $SIMTUT_DIR"
echo
ls /cvmfs/alice.cern.ch/bin/alienv && echo "CVMFS alice.cern.ch: OK"

## 2. Load O2 for the first time

The tutorial uses the tag in `swan_env.py` (`O2SIM_TAG`). To change it for all notebooks, edit that file. The first call downloads the software into the local CVMFS cache, so it can take **a few minutes**. Later calls are fast.

In [ ]:
%%bash
echo "O2_ROOT    = $O2_ROOT"
echo "O2DPG_ROOT = $O2DPG_ROOT"
which o2-sim
root-config --version

In [ ]:
%%bash
o2-sim --help | head -40

## 3. AliEn token (needed for all exercises)

When it transports particles, `o2-sim` loads the detector **alignment** from CCDB, and the O2DPG workflows read many calibration objects. Many of these objects are stored on the GRID, which needs a valid **AliEn token**.
Creating the token asks for your grid-certificate password, so it is done in a **SWAN terminal**:
*New → Terminal* in the SWAN file browser.

Your certificate must be in `~/.globus/` on CERNBox (`usercert.pem`, `userkey.pem`).
In the terminal, run the `o2token` script of this project. The path is printed by the first cell of this notebook (`project : ...`):

```bash
bash <project>/o2token
```

`o2token` prints the certificate subject and expiry date, then asks for your **grid-certificate password** once. It decrypts the key into a private file in `/tmp/$USER/globus` (file permissions cannot be enforced on CERNBox), creates the token, and **deletes the decrypted key** right away. From then on only the token is used.
The token is written to `/tmp/tokencert_<uid>.pem`, which is where the notebooks look for it. Check it here:

In [ ]:
%%bash
ls -l /tmp/tokencert_$(id -u).pem /tmp/tokenkey_$(id -u).pem 2>/dev/null || echo "no token files in /tmp: run o2token in a SWAN terminal"
alien-token-info

**If something goes wrong**
- `No certificate filename provided!` from `alien-token-info`: no valid token was found in `/tmp`. Run `o2token` again in the terminal and check that it ends with the token validity.
- `could not decrypt ... wrong password?` from `o2token`: retype the password of your grid certificate. This is not your CERN account password.
- `User certificate files NOT FOUND` or `Invalid/expired user certificate` from `o2token`: check `~/.globus/` and the expiry date printed by `o2token`.
- More details: run `ALIENPY_DEBUG=1 bash <project>/o2token` and read `~/alien_py.log`.

✅ If the token is valid, continue with **`01_o2sim_basics.ipynb`**.

> If your SWAN session is restarted, `/tmp` is wiped: create the token again. Simulation outputs in `/tmp` are also lost.
> Each notebook copies its important results to `~/simtutorial_results` on CERNBox.